# 03 — Reliability-Qualified NTL–Load Alignment

This notebook evaluates whether reliability-qualified VIIRS Black Marble DNB-BRDF nighttime lights track independent NGCP sub-grid electricity load in Samar–Leyte from 2012–2024.

The analysis moves from observability diagnostics to load alignment. Daily NTL series are aggregated using cumulative GHSL thematic masks, filtered by spatial completeness thresholds, smoothed to reduce day-level noise, normalised against NGCP load, and evaluated using association, error, and retained temporal coverage metrics.

---

In [ ]:
from pathlib import Path
import glob
import json

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr

### 1.2 Project root, input paths, and output folders

This notebook uses the standard repository output structure.

| Output type | Folder |
|---|---|
| Main manuscript figures | `outputs/figures/main/` |
| Supplementary figures | `outputs/figures/supplementary/` |
| Main manuscript tables | `outputs/tables/main/` |
| Supplementary tables | `outputs/tables/supplementary/` |

In [ ]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

NTL_GHSL_DIR = (
    PROJECT_ROOT
    / "datasets"
    / "vnp46a2"
    / "ghsl_masked_ntl_samar_leyte"
)

NGCP_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "ngcp"
    / "NGCP_Hourly_Demand.csv"
)

OUTPUT_MAIN_FIG_DIR = PROJECT_ROOT / "outputs" / "figures" / "main"
OUTPUT_SUPP_FIG_DIR = PROJECT_ROOT / "outputs" / "figures" / "supplementary"
OUTPUT_MAIN_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables" / "main"
OUTPUT_SUPP_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables" / "supplementary"

for out_dir in [
    OUTPUT_MAIN_FIG_DIR,
    OUTPUT_SUPP_FIG_DIR,
    OUTPUT_MAIN_TABLE_DIR,
    OUTPUT_SUPP_TABLE_DIR,
]:
    out_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"NTL GHSL directory: {NTL_GHSL_DIR}")
print(f"NGCP file: {NGCP_PATH}")

In [ ]:
def export_plotly_figure(
    fig,
    output_path,
    width=1200,
    height=600,
    scale=3,
    export_html=True,
    force_date_axis=False,
    tickformat="%Y",
):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fig_export = go.Figure(json.loads(pio.to_json(fig, validate=False)))

    if force_date_axis:
        fig_export.update_xaxes(type="date", tickformat=tickformat)

    fig_export.write_image(
        str(output_path.with_suffix(".png")),
        width=width,
        height=height,
        scale=scale,
    )

    fig_export.write_image(
        str(output_path.with_suffix(".svg")),
        width=width,
        height=height,
        scale=scale,
    )

    if export_html:
        fig_export.write_html(
            str(output_path.with_suffix(".html")),
            include_plotlyjs="cdn",
            full_html=True,
        )

    print(f"Exported: {output_path.with_suffix('.png')}")
    print(f"Exported: {output_path.with_suffix('.svg')}")

### 1.4 Analysis settings

In [ ]:
# ==============================
# Analysis settings
# ==============================

START_DATE = pd.Timestamp("2012-01-19")
END_DATE = pd.Timestamp("2024-12-31")
TOTAL_DAYS = (END_DATE - START_DATE).days + 1

VALUE_COL = "Mean_by_sum"
ROLLING_WINDOW = 30
CENTERED_MA = True

THRESHOLDS_MAIN = list(range(5, 100, 5))
THRESHOLDS_DISPLAY = [10, 25, 50, 75, 90]

STORM_EVENTS = [
    {"name": "Haiyan", "date": "2013-11-08"},
    {"name": "Rai", "date": "2021-12-16"},6
]

## 2. GHSL Thematic Masks

### 2.1 Cumulative GHSL mask definitions

The GHSL masks are cumulative settlement supports. They expand from the most intensely urban class to broader settlement and land classes.

| Code | GHSL range | Label |
|---|---|---|
| G1 | 30 | Urban Core |
| G2 | 23–30 | Dense Urban |
| G3 | 22–30 | Core & Inner |
| G4 | 21–30 | Urban Fringe |
| G5 | 13–30 | Built-Up / Rural Mix |
| G6 | 12–30 | Low–Mid Rural |
| G7 | 11–30 | Rural Settlements |
| G8 | 10–30 | All Land |

This design tests whether electricity-load alignment is strongest in dense, consistently illuminated urban supports and weakens as lower-density or dimmer classes are added.


In [ ]:
# ==============================
# GHSL thematic mask metadata
# ==============================

GHSL_META = {
    "30":    {"code": "G1", "label": "Urban Core",            "color": "#ff0000"},
    "23-30": {"code": "G2", "label": "Dense Urban",           "color": "#732600"},
    "22-30": {"code": "G3", "label": "Core & Inner",          "color": "#a87000"},
    "21-30": {"code": "G4", "label": "Urban Fringe",          "color": "#ffff00"},
    "13-30": {"code": "G5", "label": "Built-Up / Rural Mix",  "color": "#375623"},
    "12-30": {"code": "G6", "label": "Low–Mid Rural",         "color": "#abcd66"},
    "11-30": {"code": "G7", "label": "Rural Settlements",     "color": "#cdf57a"},
    "10-30": {"code": "G8", "label": "All Land",              "color": "#7ab6f5"},
}

GHSL_ORDER = ["30", "23-30", "22-30", "21-30", "13-30", "12-30", "11-30", "10-30"]

ghsl_mask_table = pd.DataFrame([
    {
        "Mask": GHSL_META[g]["code"],
        "GHSL Range": g,
        "Label": GHSL_META[g]["label"],
    }
    for g in GHSL_ORDER
])

ghsl_mask_table

In [ ]:
ghsl_mask_table.to_csv(
    OUTPUT_SUPP_TABLE_DIR / "table_sx_ghsl_thematic_masks_samar_leyte.csv",
    index=False,
)

## 3. Load GHSL-Masked NTL and NGCP Load Data

### 3.1 Load GHSL-masked daily NTL statistics

Each CSV represents one cumulative GHSL thematic mask. The files are combined into a single long-format panel with a `Group` column.

Expected NTL columns:

| Column | Description |
|---|---|
| `date` or `system:index` | Observation date |
| `Group` | GHSL thematic mask range |
| `Mean_by_sum` | Mask-mean DNB-BRDF radiance |
| `NTL_mean` | Alternative daily mean radiance |
| `NTL_std` | Daily within-mask standard deviation |
| `Valid_px` | Number of valid observed pixels |

In [ ]:
csv_files = sorted(glob.glob(str(NTL_GHSL_DIR / "*.csv")))

if len(csv_files) == 0:
    raise FileNotFoundError(f"No CSV files found in: {NTL_GHSL_DIR}")

df_list = []

for file in csv_files:
    file_path = Path(file)

    group_name = (
        file_path.name
        .replace("EasternVisayas_DNBBRDF_", "")
        .replace("_stats.csv", "")
    )

    temp = pd.read_csv(file_path)

    if "date" not in temp.columns:
        if "system:index" in temp.columns:
            temp["date"] = (
                temp["system:index"]
                .astype(str)
                .str.extract(r"(\d{4}_\d{2}_\d{2})")[0]
                .str.replace("_", "-", regex=False)
            )
        else:
            raise ValueError(f"No date or system:index column found in {file_path.name}")

    temp["date"] = pd.to_datetime(temp["date"], errors="coerce")
    temp["Group"] = group_name

    temp = temp[
        (temp["date"] >= START_DATE)
        & (temp["date"] <= END_DATE)
    ].copy()

    df_list.append(temp)

df_all = pd.concat(df_list, ignore_index=True)
df_all["Group"] = df_all["Group"].astype(str)

df_all = df_all.sort_values(["Group", "date"]).reset_index(drop=True)

print(f"Rows: {len(df_all):,}")
print(f"Groups: {sorted(df_all['Group'].unique())}")
print(f"Date range: {df_all['date'].min().date()} to {df_all['date'].max().date()}")

df_all.head()

In [ ]:
df_all.to_csv(
    OUTPUT_SUPP_TABLE_DIR / "table_sx_samar_leyte_ghsl_masked_ntl_panel.csv",
    index=False,
)

### 3.2 Load NGCP 1 AM electricity load

NGCP hourly load is reduced to the 1 AM value to approximate the VIIRS nighttime overpass window.

Expected NGCP columns after cleaning:

| Column | Description |
|---|---|
| `Date` | Daily date |
| `Grid_1AM_MW` | NGCP 1 AM load in megawatts |

In [ ]:
# ==============================
# Load NGCP hourly demand and extract 1 AM load
# ==============================

df_ngcp_raw = pd.read_csv(NGCP_PATH, skiprows=2)

df_ngcp_raw.columns = ["Date", "Coincident Peak", "Hour No"] + [str(i) for i in range(1, 25)]

df_ngcp = (
    df_ngcp_raw[["Date", "1"]]
    .rename(columns={"1": "Grid_1AM_MW"})
    .copy()
)

df_ngcp["Date"] = pd.to_datetime(df_ngcp["Date"], errors="coerce", dayfirst=True)
df_ngcp["Grid_1AM_MW"] = pd.to_numeric(df_ngcp["Grid_1AM_MW"], errors="coerce")

df_ngcp = df_ngcp[
    (df_ngcp["Date"] >= START_DATE)
    & (df_ngcp["Date"] <= END_DATE)
].copy()

df_ngcp = df_ngcp.sort_values("Date").dropna(subset=["Date", "Grid_1AM_MW"])

print(f"Rows: {len(df_ngcp):,}")
print(f"Date range: {df_ngcp['Date'].min().date()} to {df_ngcp['Date'].max().date()}")

df_ngcp.head()

In [ ]:
df_ngcp.to_csv(
    OUTPUT_SUPP_TABLE_DIR / "table_sx_ngcp_1am_load_samar_leyte_2012_2024.csv",
    index=False,
)

## 4. Shared Preprocessing Functions

### 4.1 Reliability-qualified NTL preprocessing

The alignment workflow uses the same preprocessing sequence for every GHSL mask and threshold:

1. Remove days with no valid observed pixels.
2. Compute spatial completeness, `SC (%)`.
3. Winsorise the NTL series at the 0–95th percentile.
4. Retain days satisfying `SC ≥ τ`.
5. Apply a 30-day moving average.
6. Align with NGCP load on overlapping dates.
7. Standardise both series using z-scores.

This keeps the comparison scale-neutral and makes the metrics reflect co-movement rather than differences in units or class brightness.

In [ ]:
# ==============================
# Event Markers
# ==============================

STORM_MARKERS = [
    ("Haiyan", "2013-11-08"),
    ("Rai", "2021-12-16"),
]


def add_event_markers(fig, y=1.04):
    for name, event_date in STORM_MARKERS:
        fig.add_vline(
            x=event_date,
            line_dash="dash",
            line_color="black",
            line_width=2,
        )

        fig.add_annotation(
            x=event_date,
            y=y,
            yref="paper",
            text=name,
            showarrow=False,
            font=dict(size=14, color="black"),
            bgcolor="white",
            bordercolor="black",
            borderwidth=1,
            xanchor="left",
        )

    return fig


def hex_to_rgba(hex_color, alpha=0.5):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"


def prepare_ntl_group(
    gdf,
    tau=10,
    value_col=VALUE_COL,
    p_lo=0,
    p_hi=95,
    w=ROLLING_WINDOW,
    centered=CENTERED_MA,
):
    ntl = gdf.copy()
    ntl = ntl[ntl["Valid_px"] > 0].copy()
    ntl["date"] = pd.to_datetime(ntl["date"])
    ntl = ntl.sort_values("date")

    if ntl.empty:
        return ntl

    total_px = ntl["Valid_px"].max()
    ntl["SC_pct"] = ntl["Valid_px"] / total_px * 100.0

    lo = np.nanpercentile(ntl[value_col], p_lo)
    hi = np.nanpercentile(ntl[value_col], p_hi)

    ntl["_val"] = np.clip(ntl[value_col], lo, hi)

    ntl = ntl[ntl["SC_pct"] >= tau].copy()

    if ntl.empty:
        return ntl

    ntl["NTL_MA"] = (
        ntl["_val"]
        .rolling(w, min_periods=1, center=centered)
        .mean()
    )

    return ntl


def prepare_ngcp_load(
    df_ngcp,
    w=ROLLING_WINDOW,
    centered=CENTERED_MA,
):
    ng = df_ngcp.copy()
    ng["Date"] = pd.to_datetime(ng["Date"])
    ng = ng.sort_values("Date")

    ng["Load_MA"] = (
        ng["Grid_1AM_MW"]
        .rolling(w, min_periods=1, center=centered)
        .mean()
    )

    return ng


def zscore(series):
    series = pd.Series(series).astype(float)
    std = series.std(ddof=0)

    if std == 0 or not np.isfinite(std):
        return pd.Series(np.nan, index=series.index)

    return (series - series.mean()) / std


## 5. Settlement-Aggregated NTL with Reliability Filter

### 5.1 Interactive diagnostic: SC filtering and smoothed NTL

This diagnostic visualises how spatial completeness filtering changes the GHSL-aggregated DNB-BRDF time series. The upper panel shows daily spatial completeness, while the lower panel shows daily radiance and the 30-day moving average after threshold filtering.

This figure is primarily exploratory and is exported as HTML only.

In [ ]:
def plot_timeseries_with_buttons(df_all, threshold=40, w=ROLLING_WINDOW):
    groups = [g for g in GHSL_ORDER if g in df_all["Group"].unique()]

    fig = make_subplots(
        rows=2,
        cols=1,
        row_heights=[0.15, 0.75],
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=("Daily spatial completeness, SC (%)", "Reliability-qualified mean DNB-BRDF"),
    )

    buttons = []
    n_traces_per_group = 3

    for i, group in enumerate(groups):
        gdf = df_all[df_all["Group"] == group].copy()
        base = prepare_ntl_group(gdf, tau=0, w=w)

        if base.empty:
            continue

        plot_df = prepare_ntl_group(gdf, tau=threshold, w=w)

        code = GHSL_META[group]["code"]
        label = GHSL_META[group]["label"]
        color = GHSL_META[group]["color"]

        fig.add_trace(
            go.Heatmap(
                z=np.expand_dims(base["SC_pct"].values, axis=0),
                x=base["date"].dt.strftime("%Y-%m-%d"),
                y=[0],
                colorscale="Greens",
                zmin=0,
                zmax=100,
                showscale=True,
                colorbar=dict(title="SC (%)", len=0.22, thickness=16, x=0.96, y=0.95),
                visible=(i == 0),
            ),
            row=1,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=plot_df["date"].dt.strftime("%Y-%m-%d"),
                y=plot_df["_val"],
                mode="markers",
                marker=dict(color=color, size=4, opacity=0.18),
                name=f"{code} daily",
                visible=(i == 0),
                showlegend=False,
            ),
            row=2,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=plot_df["date"].dt.strftime("%Y-%m-%d"),
                y=plot_df["NTL_MA"],
                mode="lines",
                line=dict(color=color, width=2.2),
                name=f"{code}: {label}",
                visible=(i == 0),
            ),
            row=2,
            col=1,
        )

        visibility = [False] * (n_traces_per_group * len(groups))
        for j in range(n_traces_per_group):
            visibility[i * n_traces_per_group + j] = True

        buttons.append(
            dict(
                label=code,
                method="update",
                args=[
                    {"visible": visibility},
                    {"title": f"Samar–Leyte DNB-BRDF NTL — {code} (SC ≥ {threshold}%)"},
                ],
            )
        )

    add_event_markers(fig, y=0.86)

    fig.update_layout(
        updatemenus=[
            dict(
                type="buttons",
                buttons=buttons,
                direction="right",
                showactive=True,
                x=0.5,
                xanchor="center",
                y=-0.4,
                yanchor="top",
            )
        ],
        xaxis2=dict(title="Date", rangeslider=dict(visible=True), type="date"),
        yaxis=dict(showticklabels=False),
        yaxis2=dict(title="Radiance (nW·cm⁻²·sr⁻¹)"),
        template="plotly_white",
        paper_bgcolor="rgba(0,0,0,0)",
        legend=dict(orientation="h", y=0.85, x=0.78, font=dict(size=14)),
        height=650,
        width=1200,
        margin=dict(t=70, l=70, b=110, r=40),
    )

    return fig


fig_sx_interactive_ntl = plot_timeseries_with_buttons(df_all, threshold=40, w=ROLLING_WINDOW)
fig_sx_interactive_ntl.show()

In [ ]:
fig_sx_interactive_ntl.write_html(
    OUTPUT_SUPP_FIG_DIR / "fig_sx_interactive_ntl_reliability_filter_samar_leyte.html",
    include_plotlyjs="cdn",
    full_html=True,
)

### 5.2 Diagnostic discussion

The reliability-filtered NTL time series shows that spatial completeness is highly intermittent across the full record. Filtering removes the weakest observation days, while the 30-day moving average preserves broad structural changes, including the post-Haiyan disruption and recovery trajectory.

This diagnostic motivates the formal threshold sweep below. The key issue is not whether a daily VNP46A2 layer exists, but whether enough of the target settlement support was directly observed to justify interpreting that day.


In [ ]:
# ==============================
# Static z-score NTL–NGCP alignment figure
# ==============================

def plot_zscore_alignment_static(
    df_all,
    df_ngcp,
    tau=10,
    w=ROLLING_WINDOW,
    value_col=VALUE_COL,
):
    groups = [g for g in GHSL_ORDER if g in df_all["Group"].unique()]
    ng = prepare_ngcp_load(df_ngcp, w=w)

    fig = go.Figure()

    for group in groups:
        gdf = df_all[df_all["Group"] == group].copy()
        ntl = prepare_ntl_group(gdf, tau=tau, value_col=value_col, w=w)

        if ntl.empty:
            continue

        merged = pd.merge(
            ntl[["date", "NTL_MA"]].rename(columns={"date": "Date"}),
            ng[["Date", "Load_MA"]],
            on="Date",
            how="inner",
        ).dropna()

        if len(merged) < 5:
            continue

        merged["NTL_z"] = zscore(merged["NTL_MA"])
        merged["Load_z"] = zscore(merged["Load_MA"])

        color = GHSL_META[group]["color"]
        code = GHSL_META[group]["code"]
        label = GHSL_META[group]["label"]

        fig.add_trace(
            go.Scatter(
                x=merged["Date"].dt.strftime("%Y-%m-%d"),
                y=merged["NTL_z"],
                mode="lines",
                line=dict(color=hex_to_rgba(color, 0.85), width=8),
                name=f"{code}: {label}" + " (NTL)",
            )
        )

    ng_plot = ng.dropna(subset=["Load_MA"]).copy()
    ng_plot["Load_z"] = zscore(ng_plot["Load_MA"])

    fig.add_trace(
        go.Scatter(
            x=ng_plot["Date"].dt.strftime("%Y-%m-%d"),
            y=ng_plot["Load_z"],
            mode="lines",
            line=dict(color="black", width=10),
            name="NGCP Electric Demand (1 AM)",
        )
    )

    add_event_markers(fig, y=1.04)

    fig.update_layout(
        template="plotly_white",
        width=1200,
        height=400,
        margin=dict(t=60, b=70, l=70, r=260),
        paper_bgcolor="rgba(0,0,0,0)",
        xaxis=dict(title="Date", type="date"),
        yaxis=dict(title="Normalised value (z-score)"),
        legend=dict(
            orientation="v",
            x=1.02,
            y=1,
            font=dict(size=13),
        ),
    )

    return fig


fig_07 = plot_zscore_alignment_static(
    df_all,
    df_ngcp,
    tau=10,
    w=ROLLING_WINDOW,
    value_col=VALUE_COL,
)

fig_07.show()

In [ ]:
export_plotly_figure(
    fig_07,
    OUTPUT_MAIN_FIG_DIR / "fig_07_samar_leyte_ntl_ngcp_zscore_tau10",
    width=1200,
    height=620,
    scale=3,
    force_date_axis=True,
    tickformat="%Y",
)

### 6.2 Main figure discussion

The z-score alignment figure shows that urban-focused masks track the broad temporal structure of NGCP load more closely than broader rural-dominant masks. The strongest correspondence is visible in the post-Haiyan decline and subsequent recovery trajectory, where urban radiance responds in the same direction as the independent electricity-load series.

The figure also shows why normalisation is necessary. Absolute NTL radiance differs strongly among settlement masks, while NGCP load is measured in megawatts. Z-scoring removes these scale differences and makes the comparison about co-movement rather than magnitude.

## 7. Supplementary Threshold Sensitivity Time Series

### 7.1 Raw radiance and NGCP load at representative thresholds

These figures show how the NTL–NGCP comparison changes under representative thresholds. They are supplementary because they illustrate the sensitivity behind the main metric surfaces.

The recommended thresholds are:

| Threshold | Interpretation |
|---|---|
| `τ = 10%` | More temporally continuous but less strict spatial support |
| `τ = 50%` | Intermediate reliability–coverage balance |
| `τ = 90%` | Strong spatial support but fragmented temporal coverage |

In [ ]:
# ==============================
# Static raw-radiance NTL–NGCP threshold figure
# ==============================

def plot_raw_alignment_static(
    df_all,
    df_ngcp,
    tau=10,
    w=ROLLING_WINDOW,
    value_col=VALUE_COL,
):
    groups = [g for g in GHSL_ORDER if g in df_all["Group"].unique()]
    ng = prepare_ngcp_load(df_ngcp, w=w)

    fig = go.Figure()

    for group in groups:
        gdf = df_all[df_all["Group"] == group].copy()
        ntl = prepare_ntl_group(gdf, tau=tau, value_col=value_col, w=w)

        if ntl.empty:
            continue

        color = GHSL_META[group]["color"]
        code = GHSL_META[group]["code"]
        label = GHSL_META[group]["label"]

        fig.add_trace(
            go.Scatter(
                x=ntl["date"].dt.strftime("%Y-%m-%d"),
                y=ntl["NTL_MA"],
                mode="lines",
                line=dict(color=color, width=2),
                name=f"{code}: {label}",
                yaxis="y1",
            )
        )

    fig.add_trace(
        go.Scatter(
            x=ng["Date"].dt.strftime("%Y-%m-%d"),
            y=ng["Load_MA"],
            mode="lines",
            line=dict(color="black", width=3),
            name="NGCP load",
            yaxis="y2",
        )
    )

    add_event_markers(fig, y=1.04)

    fig.update_layout(
        template="plotly_white",
        width=1200,
        height=620,
        margin=dict(t=70, b=70, l=70, r=270),
        paper_bgcolor="rgba(0,0,0,0)",
        xaxis=dict(title="Date", type="date"),
        yaxis=dict(title="Radiance (nW·cm⁻²·sr⁻¹)", side="left"),
        yaxis2=dict(
            title="Load (MW)",
            overlaying="y",
            side="right",
            showgrid=False,
        ),
        legend=dict(
            orientation="v",
            x=1.08,
            y=1,
            font=dict(size=13),
        ),
        title=f"GHSL-stratified DNB-BRDF and NGCP load, τ = {tau}%",
    )

    return fig
# ==============================
# Export Supplementary Figure S8 panels
# ==============================

fig_s8a = plot_raw_alignment_static(df_all, df_ngcp, tau=10)
fig_s8b = plot_raw_alignment_static(df_all, df_ngcp, tau=50)
fig_s8c = plot_raw_alignment_static(df_all, df_ngcp, tau=90)

fig_s8a.show()
fig_s8b.show()
fig_s8c.show()

In [ ]:
export_plotly_figure(
    fig_s8a,
    OUTPUT_SUPP_FIG_DIR / "fig_s8a_raw_ntl_ngcp_tau10",
    width=1200,
    height=620,
    scale=3,
    force_date_axis=True,
    tickformat="%Y",
)

export_plotly_figure(
    fig_s8b,
    OUTPUT_SUPP_FIG_DIR / "fig_s8b_raw_ntl_ngcp_tau50",
    width=1200,
    height=620,
    scale=3,
    force_date_axis=True,
    tickformat="%Y",
)

export_plotly_figure(
    fig_s8c,
    OUTPUT_SUPP_FIG_DIR / "fig_s8c_raw_ntl_ngcp_tau90",
    width=1200,
    height=620,
    scale=3,
    force_date_axis=True,
    tickformat="%Y",
)


### 7.2 Supplementary figure discussion

At low thresholds, more dates are retained and the full temporal trajectory remains visible, but weakly supported observations can contribute more strongly. At intermediate thresholds, the broad temporal pattern remains interpretable while lower-observability days are removed. At very high thresholds, the retained series becomes fragmented because the spatial-support requirement removes most daily observations.

This comparison makes the reliability–coverage trade-off visible. Increasing `τ` does not simply improve the time series; it also changes continuity, amplitude, and representation of the underlying system dynamics.

## 8. Alignment Metrics Matrix

### 8.1 Metrics computed for each GHSL mask × threshold setting

This section evaluates every GHSL mask and spatial completeness threshold combination.

For each configuration, the notebook reports:

| Metric | Description |
|---|---|
| `pearson_r` | Linear co-variation between z-scored NTL and NGCP load |
| `spearman_r` | Rank-order association |
| `R2` | Coefficient of determination between normalised series |
| `RMSE` | Root mean square error after standardisation |
| `MAE` | Mean absolute error after standardisation |
| `n_days` | Number of overlapping admissible dates |
| `coverage_pct` | Retained temporal coverage relative to the full analysis window |

In [ ]:
# ==============================
# Alignment metrics
# ==============================

def compare_once(
    ntl_group_df,
    ngcp_df,
    tau=10,
    w=ROLLING_WINDOW,
    value_col=VALUE_COL,
):
    ntl = prepare_ntl_group(
        ntl_group_df,
        tau=tau,
        value_col=value_col,
        w=w,
    )

    if ntl.empty:
        return None

    ng = prepare_ngcp_load(ngcp_df, w=w)

    merged = (
        pd.merge(
            ntl[["date", "NTL_MA"]].rename(columns={"date": "Date"}),
            ng[["Date", "Load_MA"]],
            on="Date",
            how="inner",
        )
        .dropna()
        .reset_index(drop=True)
    )

    if len(merged) < 5:
        return None

    ntl_z = zscore(merged["NTL_MA"])
    load_z = zscore(merged["Load_MA"])

    keep = ntl_z.notna() & load_z.notna()
    ntl_z = ntl_z[keep]
    load_z = load_z[keep]

    if len(ntl_z) < 5:
        return None

    pearson_r = np.corrcoef(ntl_z, load_z)[0, 1]
    spearman_val = spearmanr(ntl_z, load_z, nan_policy="omit").correlation
    rmse = np.sqrt(mean_squared_error(load_z, ntl_z))
    mae = mean_absolute_error(load_z, ntl_z)
    r2 = r2_score(load_z, ntl_z)

    return {
        "n_days": int(len(ntl_z)),
        "coverage_pct": float(len(ntl_z) / TOTAL_DAYS * 100),
        "pearson_r": float(pearson_r),
        "spearman_r": float(spearman_val),
        "R2": float(r2),
        "RMSE": float(rmse),
        "MAE": float(mae),
    }


def build_metrics_matrix(
    df_all,
    df_ngcp,
    groups=None,
    thresholds=THRESHOLDS_MAIN,
    w=ROLLING_WINDOW,
    value_col=VALUE_COL,
):
    if groups is None:
        groups = [g for g in GHSL_ORDER if g in df_all["Group"].unique()]

    rows = []

    for group in groups:
        gdf = df_all[df_all["Group"] == group].copy()

        for tau in thresholds:
            stats = compare_once(
                gdf,
                df_ngcp,
                tau=tau,
                w=w,
                value_col=value_col,
            )

            base = {
                "Group": group,
                "Mask": GHSL_META[group]["code"],
                "Mask_Label": GHSL_META[group]["label"],
                "ValidPct": tau,
            }

            if stats is None:
                rows.append({
                    **base,
                    "n_days": 0,
                    "coverage_pct": 0,
                    "pearson_r": np.nan,
                    "spearman_r": np.nan,
                    "R2": np.nan,
                    "RMSE": np.nan,
                    "MAE": np.nan,
                })
            else:
                rows.append({**base, **stats})

    return pd.DataFrame(rows)


metrics_df = build_metrics_matrix(
    df_all,
    df_ngcp,
    thresholds=THRESHOLDS_MAIN,
    w=ROLLING_WINDOW,
)

metrics_df.head()

In [ ]:
metrics_df.to_csv(
    OUTPUT_SUPP_TABLE_DIR / "table_sx_samar_leyte_alignment_metrics_matrix.csv",
    index=False,
)

## 9. Main Figure 6: Metric Surfaces

### 9.1 Heatmaps of Pearson r, RMSE, and retained temporal coverage

The heatmaps summarise the systematic GHSL mask × spatial completeness threshold sweep.

Colour scales are saturated at interpretive reference values:

| Metric | Favourable reference |
|---|---|
| Pearson `r` | `r ≥ 0.80` |
| RMSE | `RMSE ≤ 0.60` |
| Temporal coverage | `N% ≥ 50` |

These are visual references, not formal acceptance thresholds. The goal is to identify regions of favourable performance and the trade-off between alignment strength, error, and retained temporal coverage.

In [ ]:
def plot_metric_heatmap(metrics_df, metric):
    groups = [g for g in GHSL_ORDER if g in metrics_df["Group"].unique()]
    thresholds = sorted(metrics_df["ValidPct"].unique())

    pivot = (
        metrics_df
        .pivot(index="Group", columns="ValidPct", values=metric)
        .reindex(index=groups, columns=thresholds)
    )

    z = pivot.values.astype(float)

    if metric == "pearson_r":
        colorscale = "Blues_r"
        zmin, zmax = 0, 0.80
        highlight = z >= 0.80
        cbar_label = "Pearson r"
        text_format = "{:.2f}"
        colorbar_config = dict(
            title=dict(text=cbar_label, side="right"),
            len=0.85,
            tickvals=[0, 0.20, 0.40, 0.60, 0.80],
            ticktext=["0", "0.20", "0.40", "0.60", "≥0.80"],
        )

    elif metric == "RMSE":
        colorscale = "Purples"
        zmin = 0.60
        zmax = np.nanpercentile(z, 95)
        highlight = z <= 0.60
        cbar_label = "RMSE"
        text_format = "{:.2f}"
        rmse_ticks = np.linspace(zmin, zmax, 5)
        colorbar_config = dict(
            title=dict(text=cbar_label, side="right"),
            len=0.85,
            tickvals=rmse_ticks,
            ticktext=[
                "≤0.60",
                f"{rmse_ticks[1]:.2f}",
                f"{rmse_ticks[2]:.2f}",
                f"{rmse_ticks[3]:.2f}",
                f"{rmse_ticks[4]:.2f}",
            ],
        )

    elif metric == "coverage_pct":
        colorscale = "Greens_r"
        zmin, zmax = 0, 50
        highlight = z >= 50
        cbar_label = "N (%)"
        text_format = "{:.0f}"
        colorbar_config = dict(
            title=dict(text=cbar_label, side="right"),
            len=0.85,
            tickvals=[0, 10, 20, 30, 40, 50],
            ticktext=["0", "10", "20", "30", "40", "≥50"],
        )

    else:
        raise ValueError("metric must be 'pearson_r', 'RMSE', or 'coverage_pct'")

    y_labels = [
        f"{GHSL_META[g]['code']}: {GHSL_META[g]['label']}"
        for g in pivot.index
    ]

    fig = go.Figure()

    fig.add_trace(
        go.Heatmap(
            z=z,
            x=pivot.columns,
            y=y_labels,
            colorscale=colorscale,
            zmin=zmin,
            zmax=zmax,
            xgap=2,
            ygap=2,
            colorbar=colorbar_config,
            showscale=True,
        )
    )

    for i in range(z.shape[0]):
        for j in range(z.shape[1]):
            if np.isfinite(z[i, j]):
                value_text = text_format.format(z[i, j])
                if highlight[i, j]:
                    value_text = f"<b>{value_text}</b>"

                fig.add_annotation(
                    x=pivot.columns[j],
                    y=y_labels[i],
                    text=value_text,
                    showarrow=False,
                    font=dict(size=14, color="black"),
                )

    fig.update_layout(
        template="plotly_white",
        width=1500,
        height=400,
        margin=dict(l=150, r=40, t=20, b=80),
        xaxis=dict(
            title="Spatial completeness threshold, τ (%)",
            tickvals=list(range(0, 101, 20)),
            tickfont=dict(size=14),
        ),
        yaxis=dict(tickfont=dict(size=14)),
        font=dict(size=16),
        paper_bgcolor="rgba(0,0,0,0)",
    )

    return fig


fig_06a = plot_metric_heatmap(metrics_df, "pearson_r")
fig_06b = plot_metric_heatmap(metrics_df, "RMSE")
fig_06c = plot_metric_heatmap(metrics_df, "coverage_pct")

fig_06a.show()
fig_06b.show()
fig_06c.show()

In [ ]:
# ==============================
# Export Main Figure 6 panels
# ==============================

export_plotly_figure(
    fig_06a,
    OUTPUT_MAIN_FIG_DIR / "fig_06a_alignment_surface_pearson_r",
    width=1500,
    height=400,
    scale=3,
    export_html=True,
)

export_plotly_figure(
    fig_06b,
    OUTPUT_MAIN_FIG_DIR / "fig_06b_alignment_surface_rmse",
    width=1500,
    height=400,
    scale=3,
    export_html=True,
)

export_plotly_figure(
    fig_06c,
    OUTPUT_MAIN_FIG_DIR / "fig_06c_alignment_surface_temporal_coverage",
    width=1500,
    height=400,
    scale=3,
    export_html=True,
)

### 9.2 Main figure discussion

The metric surfaces show that alignment is jointly controlled by settlement support and spatial completeness threshold. Urban-focused masks, especially G1–G3, produce the strongest NTL–load correspondence. Pearson correlation remains high across a broad low- to mid-threshold range, while broader rural-dominant masks generally show weaker correlation and higher error.

The retained temporal coverage surface shows the cost of increasing `τ`. As the threshold becomes stricter, fewer dates remain available for alignment. The surfaces therefore do not identify a single threshold that simultaneously maximises association, minimises error, and preserves temporal coverage. Instead, they show a reliability–coverage region where urban-focused masks and moderate spatial support provide the most defensible operating conditions.


## 10. Composite Score and Top Configurations

### 10.1 Composite score

The composite score summarises the trade-off among association strength, error, and retained temporal coverage:

$ S = 0.6\tilde{r} + 0.1\widetilde{RMSE}_{inv} + 0.3\tilde{N} $ 

All components are min–max scaled within Samar–Leyte. The RMSE component is inverted so that higher values indicate lower error.

In [ ]:
# ==============================
# Composite score for Samar–Leyte
# ==============================

def minmax_scale(s):
    s = pd.Series(s).astype(float)
    span = s.max() - s.min()

    if span == 0 or not np.isfinite(span):
        return pd.Series(0.0, index=s.index)

    return (s - s.min()) / span


rank_df = metrics_df.copy()
rank_df = rank_df.dropna(subset=["pearson_r", "RMSE", "coverage_pct"]).copy()

rank_df["r_scaled"] = minmax_scale(rank_df["pearson_r"])
rank_df["rmse_inv_scaled"] = minmax_scale(-rank_df["RMSE"])
rank_df["coverage_scaled"] = minmax_scale(rank_df["coverage_pct"])

rank_df["Composite"] = (
    0.6 * rank_df["r_scaled"]
    + 0.1 * rank_df["rmse_inv_scaled"]
    + 0.3 * rank_df["coverage_scaled"]
)

top_configs = (
    rank_df
    .sort_values("Composite", ascending=False)
    .loc[
        :,
        [
            "Mask",
            "Mask_Label",
            "Group",
            "ValidPct",
            "Composite",
            "pearson_r",
            "spearman_r",
            "R2",
            "RMSE",
            "MAE",
            "coverage_pct",
            "n_days",
        ],
    ]
    .head(10)
    .round({
        "Composite": 3,
        "pearson_r": 2,
        "spearman_r": 2,
        "R2": 2,
        "RMSE": 2,
        "MAE": 2,
        "coverage_pct": 2,
    })
)

top_configs

In [ ]:
top_configs.to_csv(
    OUTPUT_SUPP_TABLE_DIR / "table_sx_samar_leyte_top_alignment_configurations.csv",
    index=False,
)

### 10.2 Table discussion

The top configurations identify the mask–threshold settings that best balance association, error, and temporal representation for Samar–Leyte. Urban-focused masks dominate the ranking, confirming that the NTL–load relationship is strongest where nighttime radiance is anchored by persistent anthropogenic lighting.

The score should be interpreted as a within-region ranking rather than an absolute reliability measure. A high score means that the configuration performs well relative to the other GHSL mask and threshold combinations tested for Samar–Leyte.

## 11. Spatial Completeness Distribution and Retention Curves

### 11.1 SC distribution profiles

This figure shows how daily spatial completeness is distributed across GHSL masks. It explains why increasing `τ` affects groups differently.

In [ ]:
def plot_sc_distribution_profiles(
    df_all,
    thresholds=[25, 50, 75, 90],
    bin_size=10,
    value_col=VALUE_COL,
):
    groups = [g for g in GHSL_ORDER if g in df_all["Group"].unique()]
    bins = np.arange(0, 101, bin_size)
    bin_centers = (bins[:-1] + bins[1:]) / 2

    fig = go.Figure()

    for group in groups:
        gdf = prepare_ntl_group(
            df_all[df_all["Group"] == group].copy(),
            tau=0,
            value_col=value_col,
        )

        if gdf.empty:
            continue

        vals = gdf["SC_pct"].dropna()

        counts, _ = np.histogram(vals, bins=bins)
        percents = counts / counts.sum() * 100 if counts.sum() > 0 else np.zeros_like(counts)

        fig.add_trace(
            go.Scatter(
                x=bin_centers,
                y=percents,
                mode="lines+markers",
                name=f"{GHSL_META[group]['code']}: {GHSL_META[group]['label']}",
                line=dict(color=GHSL_META[group]["color"], width=3),
                marker=dict(size=7, color=GHSL_META[group]["color"]),
            )
        )

    for tau in thresholds:
        fig.add_vline(x=tau, line_dash="dash", line_color="black", line_width=1.5)

    fig.update_layout(
        template="plotly_white",
        xaxis=dict(
            title="Spatial completeness, SC (%)",
            range=[0, 100],
        ),
        yaxis=dict(
            title="Percent of days",
            range=[0, 30],
        ),
        legend=dict(
            orientation="v",
            x=1.02,
            y=1,
            font=dict(size=13),
        ),
        width=1200,
        height=550,
        margin=dict(t=50, l=70, b=70, r=280),
        paper_bgcolor="rgba(0,0,0,0)",
    )

    return fig


fig_sx_sc_dist = plot_sc_distribution_profiles(df_all)
fig_sx_sc_dist.show()

In [ ]:
export_plotly_figure(
    fig_sx_sc_dist,
    OUTPUT_SUPP_FIG_DIR / "fig_sx_sc_distribution_profiles_samar_leyte",
    width=1200,
    height=550,
    scale=3,
)

### 11.2 Cumulative retention curves

This figure shows the percentage of days retained as the spatial completeness threshold increases.


In [ ]:
# ==============================
# Cumulative retention curves
# ==============================

def plot_sc_retention_curves(
    df_all,
    thresholds=[25, 50, 75, 90],
    value_col=VALUE_COL,
):
    groups = [g for g in GHSL_ORDER if g in df_all["Group"].unique()]
    tau_range = np.arange(0, 101, 1)

    fig = go.Figure()

    for group in groups:
        gdf = prepare_ntl_group(
            df_all[df_all["Group"] == group].copy(),
            tau=0,
            value_col=value_col,
        )

        if gdf.empty:
            continue

        vals = gdf["SC_pct"].dropna().values
        retained = [(vals >= tau).sum() / len(vals) * 100 for tau in tau_range]

        fig.add_trace(
            go.Scatter(
                x=tau_range,
                y=retained,
                mode="lines",
                name=f"{GHSL_META[group]['code']}: {GHSL_META[group]['label']}",
                line=dict(color=GHSL_META[group]["color"], width=3),
                hovertemplate="τ: %{x:.0f}%<br>Retained days: %{y:.1f}%<extra></extra>",
            )
        )

    for tau in thresholds:
        fig.add_vline(x=tau, line_dash="dash", line_color="black", line_width=1.5)

    fig.update_layout(
        template="plotly_white",
        xaxis=dict(
            title="Spatial completeness threshold, τ (%)",
            range=[0, 100],
            dtick=10,
        ),
        yaxis=dict(
            title="Days retained (%)",
            range=[0, 105],
        ),
        legend=dict(
            orientation="v",
            x=1.02,
            y=1,
            font=dict(size=13),
        ),
        width=1200,
        height=600,
        margin=dict(t=50, l=70, b=70, r=280),
        paper_bgcolor="rgba(0,0,0,0)",
    )

    return fig


fig_sx_retention = plot_sc_retention_curves(df_all)
fig_sx_retention.show()

In [ ]:
export_plotly_figure(
    fig_sx_retention,
    OUTPUT_SUPP_FIG_DIR / "fig_sx_sc_retention_curves_samar_leyte",
    width=1200,
    height=600,
    scale=3,
)


### 11.3 Supplementary discussion

The SC distribution and retention curves show why threshold choice must be treated as a tunable reliability parameter. Coverage declines monotonically as `τ` increases, but the rate of decline differs among settlement supports.

Urban-focused masks retain more high-completeness days, while broader rural-dominant masks lose continuity more quickly. This explains why stricter thresholds can improve statistical alignment for some configurations while also reducing temporal representation.


## 12. Notebook Outputs

### 12.1 Main manuscript outputs

| Output | Path |
|---|---|
| Figure 6a — Pearson r alignment surface | `outputs/figures/main/fig_06a_alignment_surface_pearson_r.*` |
| Figure 6b — RMSE alignment surface | `outputs/figures/main/fig_06b_alignment_surface_rmse.*` |
| Figure 6c — Retained temporal coverage surface | `outputs/figures/main/fig_06c_alignment_surface_temporal_coverage.*` |
| Figure 7 — Samar–Leyte z-score NTL–NGCP alignment at τ = 10% | `outputs/figures/main/fig_07_samar_leyte_ntl_ngcp_zscore_tau10.*` |

### 12.2 Supplementary outputs

| Output | Path |
|---|---|
| GHSL thematic mask table | `outputs/tables/supplementary/table_sx_ghsl_thematic_masks_samar_leyte.csv` |
| GHSL-masked NTL daily panel | `outputs/tables/supplementary/table_sx_samar_leyte_ghsl_masked_ntl_panel.csv` |
| NGCP 1 AM load table | `outputs/tables/supplementary/table_sx_ngcp_1am_load_samar_leyte_2012_2024.csv` |
| Alignment metrics matrix | `outputs/tables/supplementary/table_sx_samar_leyte_alignment_metrics_matrix.csv` |
| Top Samar–Leyte configurations | `outputs/tables/supplementary/table_sx_samar_leyte_top_alignment_configurations.csv` |
| Interactive reliability-filter NTL diagnostic | `outputs/figures/supplementary/fig_sx_interactive_ntl_reliability_filter_samar_leyte.html` |
| Figure S8a — Raw NTL–NGCP alignment, τ = 10% | `outputs/figures/supplementary/fig_s8a_raw_ntl_ngcp_tau10.*` |
| Figure S8b — Raw NTL–NGCP alignment, τ = 50% | `outputs/figures/supplementary/fig_s8b_raw_ntl_ngcp_tau50.*` |
| Figure S8c — Raw NTL–NGCP alignment, τ = 90% | `outputs/figures/supplementary/fig_s8c_raw_ntl_ngcp_tau90.*` |
| SC distribution profiles | `outputs/figures/supplementary/fig_sx_sc_distribution_profiles_samar_leyte.*` |
| SC retention curves | `outputs/figures/supplementary/fig_sx_sc_retention_curves_samar_leyte.*` |

---

---

## 13. Summary

This notebook establishes the load-alignment gate of the reliability-qualified NTL workflow. The results show that NTL–load alignment depends on both settlement support and spatial completeness threshold.

Urban-focused masks produce the strongest alignment with NGCP load because their radiance signal is more directly tied to persistent electrified settlements. Broader rural-dominant masks retain a wider spatial footprint but dilute the load-related signal and generally increase error. At the same time, increasingly strict spatial completeness thresholds reduce retained temporal coverage, creating a direct trade-off between observational reliability and temporal representation.

The main conclusion is that reliability cannot be defined by a single universal threshold. Instead, reliable use of daily NTL requires evaluating the mask–threshold operating region where association, error, and retained coverage are jointly acceptable. This motivates the next notebook, which tests whether the selected reliability-qualified configurations transfer across other Visayas sub-grids.
